In [31]:
import pandas as pd
import torch
import torch.nn as nn
import tiktoken

from sklearn.model_selection import train_test_split

In [32]:
df = pd.read_csv("IMDB Dataset.csv")

print(df.head())
print(df.shape)

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
(50000, 2)


In [33]:
df = df.sample(n=3000, random_state=42)

print("Dataset size:", df.shape)

Dataset size: (3000, 2)


In [34]:
df["sentiment"] = df["sentiment"].map({
    "positive": 1,
    "negative": 0
})

print(df.head())

                                                  review  sentiment
33553  I really liked this Summerslam due to the look...          1
9427   Not many television shows appeal to quite as m...          1
199    The film quickly gets to a major chase scene w...          0
12447  Jane Austen would definitely approve of this o...          1
39489  Expectations were somewhat high for me when I ...          0


In [35]:
reviews = df["review"].tolist()
labels = df["sentiment"].tolist()

print("Number of reviews:", len(reviews))
print("Number of labels:", len(labels))

Number of reviews: 3000
Number of labels: 3000


In [36]:
X_train, X_test, y_train, y_test = train_test_split(
    reviews,
    labels,
    test_size=0.2,
    random_state=42
)

print("Training reviews:", len(X_train))
print("Testing reviews:", len(X_test))

Training reviews: 2400
Testing reviews: 600


In [37]:
tokenizer = tiktoken.get_encoding("cl100k_base")

print("Vocabulary size:", tokenizer.n_vocab)

Vocabulary size: 100277


In [38]:
def tokenize_text(text):
    tokens = tokenizer.encode(text)

    # Add 1 so that 0 can be used for padding
    tokens = [token + 1 for token in tokens]

    return tokens

In [39]:
X_train_tokens = [tokenize_text(text) for text in X_train]
X_test_tokens = [tokenize_text(text) for text in X_test]

print(X_train_tokens[0][:20])

[73, 1098, 265, 2467, 8572, 316, 24319, 820, 1765, 82470, 324, 25429, 17873, 14, 359, 15747, 421, 4633, 1307, 603]


In [40]:
MAX_LEN = 100

def pad_sequence(tokens, max_len):

    # Truncate if the review is too long
    if len(tokens) > max_len:
        tokens = tokens[:max_len]

    # Pad with 0 if the review is too short
    while len(tokens) < max_len:
        tokens.append(0)

    return tokens

In [41]:
X_train_padded = [
    pad_sequence(tokens, MAX_LEN)
    for tokens in X_train_tokens
]

X_test_padded = [
    pad_sequence(tokens, MAX_LEN)
    for tokens in X_test_tokens
]

print("Length of one review:", len(X_train_padded[0]))

Length of one review: 100


In [42]:
X_train_tensor = torch.tensor(X_train_padded, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_padded, dtype=torch.long)

y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

print("X_train shape:", X_train_tensor.shape)
print("y_train shape:", y_train_tensor.shape)

X_train shape: torch.Size([2400, 100])
y_train shape: torch.Size([2400])


In [43]:
class RNNClassifier(nn.Module):

    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super().__init__()

        # Embedding layer
        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        # RNN layer
        self.rnn = nn.RNN(
            embedding_dim,
            hidden_dim,
            batch_first=True
        )

        # Output layer
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):

        # Convert token IDs into embeddings
        embedded = self.embedding(x)

        # Pass embeddings through RNN
        output, hidden = self.rnn(embedded)

        # Take the last output
        last_output = output[:, -1, :]

        # Final prediction
        output = self.fc(last_output)

        return output

In [44]:
vocab_size = tokenizer.n_vocab + 1
embedding_dim = 64
hidden_dim = 64

model = RNNClassifier(
    vocab_size,
    embedding_dim,
    hidden_dim
)

print(model)

RNNClassifier(
  (embedding): Embedding(100278, 64, padding_idx=0)
  (rnn): RNN(64, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)


In [45]:
loss_function = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [48]:
epochs = 5

for epoch in range(epochs):

    model.train()

    optimizer.zero_grad()

    # Forward pass
    outputs = model(X_train_tensor)

    # Remove extra dimension
    outputs = outputs.squeeze()

    # Calculate loss
    loss = loss_function(
        outputs,
        y_train_tensor
    )

    # Backpropagation
    loss.backward()

    # Update weights
    optimizer.step()

    print(
        "Epoch:",
        epoch + 1,
        "Loss:",
        loss.item()
    )

Epoch: 1 Loss: 0.68458092212677
Epoch: 2 Loss: 0.6817572712898254
Epoch: 3 Loss: 0.6790052056312561
Epoch: 4 Loss: 0.6762855052947998
Epoch: 5 Loss: 0.6735627055168152
